# Phase 1-2 Re-Validation (v66 Fixes)

**Two foundational bugs fixed:**

1. **ORB State Machine** — Breakout detection now latched, not re-evaluated every bar
2. **Stop Validity** — VWAP/Swing stops validated for wrong-side and min distance (15% ATR)

This notebook validates Python matches Pine v66.

In [1]:
# Cell 1: Setup
import sys
sys.path.insert(0, r'C:\Users\phemm\orb_lab\src')

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv(r'C:\Users\phemm\orb_lab\.env')

from data_collector import PolygonDataCollector

# Fetch with enough history for warmup
collector = PolygonDataCollector()
df = collector.fetch_bars('NVDA', days_back=120, bar_size=1)
print(f"✓ Loaded {len(df)} bars")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")

[DataCollector] Using cache: C:\Users\phemm\orb_lab\data\cache
📊 Fetching NVDA (1-min bars, 120 days)...
  ⚠ Cache expired (20.9 hours old), refreshing...
  ✓ Fetched 32095 bars, cached for next time
✓ Loaded 32095 bars
Date range: 2025-09-02 to 2025-12-29


In [2]:
# Cell 2: Basic calculations (ATR, VWAP)

def calc_atr(df, period=14):
    """ATR using Wilder's smoothing (RMA)"""
    df = df.copy()
    tr1 = df['high'] - df['low']
    tr2 = abs(df['high'] - df['close'].shift(1))
    tr3 = abs(df['low'] - df['close'].shift(1))
    df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['atr'] = df['tr'].ewm(alpha=1/period, adjust=False).mean()
    return df

def calc_session_vwap(df):
    """Session VWAP resetting at 9:30"""
    df = df.copy()
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    df['tp_volume'] = df['typical_price'] * df['volume']
    df['new_day'] = (df.index.hour == 9) & (df.index.minute == 30)
    df['day_group'] = df['new_day'].cumsum()
    df['cum_tp_vol'] = df.groupby('day_group')['tp_volume'].cumsum()
    df['cum_vol'] = df.groupby('day_group')['volume'].cumsum()
    df['vwap'] = df['cum_tp_vol'] / df['cum_vol']
    return df

df = calc_atr(df)
df = calc_session_vwap(df)
print("✓ ATR and VWAP calculated")

✓ ATR and VWAP calculated


In [3]:
# Cell 3: ORB State Machine (v66 Fix #1)

def detect_orb_with_state_machine(df, orb_minutes=5, 
                                   breakout_threshold_mult=0.1,
                                   min_body_strength=0.5,
                                   require_open_in_range=True):
    """
    Pine v66 State Machine:
    1. DETECT: Only on actual breakout candle (all criteria must pass)
    2. LATCH: Set pending=True when detected
    3. USE: baseBreakout = pending
    4. RESET: When price returns inside ORB
    """
    df = df.copy()
    
    # ORB Session detection
    def is_in_orb(ts):
        bar_total = ts.hour * 60 + ts.minute
        return 9*60+30 <= bar_total < 9*60+30+orb_minutes
    
    df['in_session'] = df.index.map(is_in_orb)
    df['is_new_session'] = df['in_session'] & ~df['in_session'].shift(1, fill_value=False)
    
    # Calculate ORB levels
    df['orb_high'] = np.nan
    df['orb_low'] = np.nan
    current_high, current_low = np.nan, np.nan
    
    for i in range(len(df)):
        if df['is_new_session'].iloc[i]:
            current_high = df['high'].iloc[i]
            current_low = df['low'].iloc[i]
        elif df['in_session'].iloc[i]:
            current_high = max(current_high, df['high'].iloc[i])
            current_low = min(current_low, df['low'].iloc[i])
        df.iloc[i, df.columns.get_loc('orb_high')] = current_high
        df.iloc[i, df.columns.get_loc('orb_low')] = current_low
    
    df['orb_complete'] = ~df['in_session'] & df['orb_high'].notna() & df['orb_low'].notna()
    
    # Breakout quality checks (evaluated on each bar for detection)
    df['breakout_threshold'] = df['atr'] * breakout_threshold_mult
    df['sufficient_high'] = df['close'] > (df['orb_high'] + df['breakout_threshold'])
    df['sufficient_low'] = df['close'] < (df['orb_low'] - df['breakout_threshold'])
    
    df['candle_body'] = abs(df['close'] - df['open'])
    df['candle_range'] = df['high'] - df['low']
    df['body_ratio'] = df['candle_body'] / df['candle_range'].replace(0, np.nan)
    df['strong_body'] = df['body_ratio'] >= min_body_strength
    
    # STATE MACHINE - iterate through bars
    df['long_breakout_pending'] = False
    df['short_breakout_pending'] = False
    df['has_broken_out_high'] = False
    df['has_broken_out_low'] = False
    df['new_long_breakout'] = False
    df['new_short_breakout'] = False
    
    long_pending = False
    short_pending = False
    has_broken_high = False
    has_broken_low = False
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        # Reset at new session
        if row['is_new_session']:
            long_pending = False
            short_pending = False
            has_broken_high = False
            has_broken_low = False
        
        orb_complete = row['orb_complete']
        orb_high = row['orb_high']
        orb_low = row['orb_low']
        
        # 1. DETECT new breakouts
        new_long = False
        new_short = False
        
        if orb_complete:
            open_price = row['open']
            close_price = row['close']
            sufficient_high = row['sufficient_high']
            sufficient_low = row['sufficient_low']
            strong_body = row['strong_body']
            
            if require_open_in_range:
                # Strict: open must be inside range
                open_in_range = (open_price >= orb_low) and (open_price <= orb_high)
                new_long = open_in_range and (close_price > orb_high) and sufficient_high and strong_body
                new_short = open_in_range and (close_price < orb_low) and sufficient_low and strong_body
            else:
                # Relaxed: just close outside with quality
                new_long = (close_price > orb_high) and sufficient_high and strong_body
                new_short = (close_price < orb_low) and sufficient_low and strong_body
        
        # 2. LATCH
        if new_long and not has_broken_high:
            long_pending = True
        if new_short and not has_broken_low:
            short_pending = True
        
        # 4. RESET when price returns inside ORB
        if orb_complete:
            if long_pending and row['close'] <= orb_high:
                long_pending = False
            if short_pending and row['close'] >= orb_low:
                short_pending = False
            if has_broken_high and row['close'] <= orb_high:
                has_broken_high = False
            if has_broken_low and row['close'] >= orb_low:
                has_broken_low = False
        
        # Store state
        df.iloc[i, df.columns.get_loc('new_long_breakout')] = new_long
        df.iloc[i, df.columns.get_loc('new_short_breakout')] = new_short
        df.iloc[i, df.columns.get_loc('long_breakout_pending')] = long_pending
        df.iloc[i, df.columns.get_loc('short_breakout_pending')] = short_pending
        df.iloc[i, df.columns.get_loc('has_broken_out_high')] = has_broken_high
        df.iloc[i, df.columns.get_loc('has_broken_out_low')] = has_broken_low
    
    # 3. USE - baseBreakout = pending
    df['base_breakout_high'] = df['long_breakout_pending']
    df['base_breakout_low'] = df['short_breakout_pending']
    
    return df

# Apply state machine with require_open_in_range=True (the fix)
df = detect_orb_with_state_machine(df, require_open_in_range=True)
print("✓ ORB state machine applied")
print(f"New long breakouts detected: {df['new_long_breakout'].sum()}")
print(f"New short breakouts detected: {df['new_short_breakout'].sum()}")

✓ ORB state machine applied
New long breakouts detected: 201
New short breakouts detected: 215


In [4]:
# Cell 4: Stop Validity Checks (v66 Fix #2)

def find_swing_low_rth(df, idx, lookback=5):
    """Find swing low using only RTH bars"""
    swing_low = None
    valid_bars = 0
    for i in range(1, lookback * 3 + 1):
        if idx - i < 0:
            break
        bar_time = df.index[idx - i]
        bar_minutes = bar_time.hour * 60 + bar_time.minute
        if 570 <= bar_minutes < 960:  # 9:30 to 16:00
            bar_low = df['low'].iloc[idx - i]
            if swing_low is None or bar_low < swing_low:
                swing_low = bar_low
            valid_bars += 1
            if valid_bars >= lookback:
                break
    return swing_low if swing_low else df['low'].iloc[max(0, idx-1)]

def find_swing_high_rth(df, idx, lookback=5):
    """Find swing high using only RTH bars"""
    swing_high = None
    valid_bars = 0
    for i in range(1, lookback * 3 + 1):
        if idx - i < 0:
            break
        bar_time = df.index[idx - i]
        bar_minutes = bar_time.hour * 60 + bar_time.minute
        if 570 <= bar_minutes < 960:
            bar_high = df['high'].iloc[idx - i]
            if swing_high is None or bar_high > swing_high:
                swing_high = bar_high
            valid_bars += 1
            if valid_bars >= lookback:
                break
    return swing_high if swing_high else df['high'].iloc[max(0, idx-1)]


def calculate_stops_with_validity(df, idx, is_long,
                                   atr_mult=2.0, swing_lookback=5, swing_buffer=0.02,
                                   vwap_distance=0.3, min_stop_atr=0.15):
    """
    Pine v66 Stop Validity:
    - VWAP must be on correct side of entry
    - All stops must be at least 15% ATR from entry
    - Invalid stops fallback to ATR
    """
    entry = df['close'].iloc[idx]
    atr = df['atr'].iloc[idx]
    vwap = df['vwap'].iloc[idx]
    
    min_stop_dist = atr * min_stop_atr
    
    # Calculate raw stops
    if is_long:
        atr_stop = entry - (atr * atr_mult)
        swing_low = find_swing_low_rth(df, idx, swing_lookback)
        swing_stop = swing_low - (atr * swing_buffer)
        vwap_stop = vwap - (atr * vwap_distance)
    else:
        atr_stop = entry + (atr * atr_mult)
        swing_high = find_swing_high_rth(df, idx, swing_lookback)
        swing_stop = swing_high + (atr * swing_buffer)
        vwap_stop = vwap + (atr * vwap_distance)
    
    # VALIDITY CHECKS
    if is_long:
        atr_valid = atr_stop < entry - min_stop_dist
        swing_valid = swing_stop < entry - min_stop_dist
        # VWAP must be BELOW entry AND stop far enough below
        vwap_valid = (vwap < entry) and (vwap_stop < entry - min_stop_dist)
    else:
        atr_valid = atr_stop > entry + min_stop_dist
        swing_valid = swing_stop > entry + min_stop_dist
        # VWAP must be ABOVE entry AND stop far enough above
        vwap_valid = (vwap > entry) and (vwap_stop > entry + min_stop_dist)
    
    # Fallback invalid stops to ATR
    if not swing_valid:
        swing_stop = atr_stop
    if not vwap_valid:
        vwap_stop = atr_stop
    
    # Hybrid = tightest valid stop (closest to entry while still valid)
    if is_long:
        valid_stops = [atr_stop]  # ATR always valid as baseline
        if swing_valid:
            valid_stops.append(swing_stop)
        if vwap_valid:
            valid_stops.append(vwap_stop)
        hybrid_stop = max(valid_stops)  # Tightest = highest for long
    else:
        valid_stops = [atr_stop]
        if swing_valid:
            valid_stops.append(swing_stop)
        if vwap_valid:
            valid_stops.append(vwap_stop)
        hybrid_stop = min(valid_stops)  # Tightest = lowest for short
    
    return {
        'entry': entry,
        'atr': atr,
        'vwap': vwap,
        'min_stop_dist': min_stop_dist,
        'atr_stop': atr_stop,
        'swing_stop': swing_stop,
        'vwap_stop': vwap_stop,
        'hybrid_stop': hybrid_stop,
        'atr_valid': atr_valid,
        'swing_valid': swing_valid,
        'vwap_valid': vwap_valid
    }

print("✓ Stop validity functions defined")

✓ Stop validity functions defined


In [8]:
# Cell 5: R:R Calculation with validity

def calculate_best_rr_with_validity(stops, is_long, rr_desired=2.0, max_target_atr=3.0):
    """
    Pine v66: Only valid stops get R:R calculated.
    Invalid stops get R:R = -1 (excluded from selection).
    """
    entry = stops['entry']
    atr = stops['atr']
    
    results = {}
    
    for stop_type in ['atr', 'swing', 'vwap', 'hybrid']:
        stop_key = f'{stop_type}_stop'
        valid_key = f'{stop_type}_valid' if stop_type != 'hybrid' else None
        
        stop_price = stops[stop_key]
        
        # Check validity (hybrid validity = based on components)
        if stop_type == 'hybrid':
            is_valid = True  # Hybrid is constructed from valid stops
        else:
            is_valid = stops.get(valid_key, True)
        
        if not is_valid:
            results[stop_type] = {'stop': stop_price, 'risk': 0, 'rr': -1, 'valid': False}
            continue
        
        risk = abs(entry - stop_price)
        if risk <= 0:
            results[stop_type] = {'stop': stop_price, 'risk': 0, 'rr': -1, 'valid': False}
            continue
        
        # Target at desired R:R
        target_distance = risk * rr_desired
        target_atrs = target_distance / atr
        
        # Cap R:R if target too far
        if target_atrs > max_target_atr:
            best_rr = (max_target_atr * atr) / risk
        else:
            best_rr = rr_desired
        
        results[stop_type] = {
            'stop': stop_price,
            'risk': risk,
            'target_atrs': target_atrs,
            'rr': best_rr,
            'valid': True
        }
    
    return results


def select_best_valid_stop(rr_results):
    """
    Select stop with highest R:R among VALID stops only.
    Fallback to ATR if nothing else valid.
    """
    best_type = 'atr'  # Default fallback
    best_rr = rr_results['atr']['rr'] if rr_results['atr']['valid'] else 0
    
    for stop_type, data in rr_results.items():
        if data['valid'] and data['rr'] > best_rr:
            best_rr = data['rr']
            best_type = stop_type
    
    return best_type, best_rr

print("✓ R:R calculation with validity defined")

✓ R:R calculation with validity defined


In [5]:
# Cell 6: Test state machine on Oct 29 (valid breakout that should execute)

test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"=== State Machine Test: {test_date} ===")
print(f"ORB High: {day_df['orb_high'].iloc[-1]:.2f}")
print(f"ORB Low: {day_df['orb_low'].iloc[-1]:.2f}")

# Find breakout detection
breakouts = day_df[day_df['new_long_breakout']]
if len(breakouts) > 0:
    first_breakout = breakouts.iloc[0]
    ts = breakouts.index[0]
    print(f"\n📍 NEW LONG BREAKOUT detected at {ts.strftime('%H:%M')}")
    print(f"   Open:  {first_breakout['open']:.2f}")
    print(f"   Close: {first_breakout['close']:.2f}")
    print(f"   ORB High: {first_breakout['orb_high']:.2f}")
    print(f"   Open in range: {first_breakout['open'] >= first_breakout['orb_low'] and first_breakout['open'] <= first_breakout['orb_high']}")
    print(f"   Sufficient threshold: {first_breakout['sufficient_high']}")
    print(f"   Strong body: {first_breakout['strong_body']} (ratio: {first_breakout['body_ratio']:.2f})")
    
    # Check next bar (entry bar)
    next_idx = day_df.index.get_loc(ts) + 1
    if next_idx < len(day_df):
        entry_bar = day_df.iloc[next_idx]
        print(f"\n📍 ENTRY BAR at {day_df.index[next_idx].strftime('%H:%M')}")
        print(f"   Open: {entry_bar['open']:.2f} (this is entry price)")
        print(f"   Pending still active: {entry_bar['long_breakout_pending']}")
else:
    print("\n❌ No new long breakout detected")
    # Debug: show what bars looked like breakouts but failed
    potential = day_df[day_df['orb_complete'] & (day_df['close'] > day_df['orb_high'])]
    if len(potential) > 0:
        print(f"\nBars that closed above ORB but failed breakout criteria:")
        for ts, row in potential.head(3).iterrows():
            open_in_range = row['open'] >= row['orb_low'] and row['open'] <= row['orb_high']
            print(f"   {ts.strftime('%H:%M')}: open_in_range={open_in_range}, sufficient={row['sufficient_high']}, strong_body={row['strong_body']}")

=== State Machine Test: 2025-10-29 ===
ORB High: 211.08
ORB Low: 207.09

📍 NEW LONG BREAKOUT detected at 09:59
   Open:  211.01
   Close: 211.97
   ORB High: 211.08
   Open in range: True
   Sufficient threshold: True
   Strong body: True (ratio: 0.94)

📍 ENTRY BAR at 10:00
   Open: 211.97 (this is entry price)
   Pending still active: True


In [6]:
# Cell 7: Test stop validity on a scenario where VWAP is wrong-side

# Find a bar where VWAP > entry for a LONG (would have caused the bug)
test_date = '2025-10-29'
day_df = df[df.index.date.astype(str) == test_date].copy()

# Find the breakout bar
breakouts = day_df[day_df['new_long_breakout']]
if len(breakouts) > 0:
    ts = breakouts.index[0]
    idx = df.index.get_loc(ts)
    
    # Use next bar for entry (realistic)
    entry_idx = idx + 1
    
    print(f"=== Stop Validity Test: {test_date} ===")
    
    stops = calculate_stops_with_validity(df, entry_idx, is_long=True)
    
    print(f"\nEntry: {stops['entry']:.2f}")
    print(f"VWAP:  {stops['vwap']:.2f}")
    print(f"ATR:   {stops['atr']:.4f}")
    print(f"Min Stop Dist (15% ATR): {stops['min_stop_dist']:.4f}")
    
    print(f"\n--- Stop Prices ---")
    print(f"ATR Stop:    {stops['atr_stop']:.2f} (valid: {stops['atr_valid']})")
    print(f"Swing Stop:  {stops['swing_stop']:.2f} (valid: {stops['swing_valid']})")
    print(f"VWAP Stop:   {stops['vwap_stop']:.2f} (valid: {stops['vwap_valid']})")
    print(f"Hybrid Stop: {stops['hybrid_stop']:.2f}")
    
    # Check if VWAP would have been invalid
    if stops['vwap'] >= stops['entry']:
        print(f"\n⚠️  VWAP ({stops['vwap']:.2f}) >= Entry ({stops['entry']:.2f})")
        print(f"    In old code: Would create 1-penny stop → 10k shares!")
        print(f"    With fix: VWAP invalid, fallback to ATR")
    else:
        print(f"\n✓ VWAP ({stops['vwap']:.2f}) < Entry ({stops['entry']:.2f}) - valid scenario")

=== Stop Validity Test: 2025-10-29 ===

Entry: 211.98
VWAP:  210.13
ATR:   0.6517
Min Stop Dist (15% ATR): 0.0978

--- Stop Prices ---
ATR Stop:    210.68 (valid: True)
Swing Stop:  209.79 (valid: True)
VWAP Stop:   209.94 (valid: True)
Hybrid Stop: 210.68

✓ VWAP (210.13) < Entry (211.98) - valid scenario


In [9]:
# Cell 8: Test R:R gate with validity

test_date = '2025-10-30'  # The SKIP trade
day_df = df[df.index.date.astype(str) == test_date].copy()

print(f"=== R:R Gate Test: {test_date} (SKIP trade) ===")

# Find SHORT breakout
breakouts = day_df[day_df['new_short_breakout']]
if len(breakouts) > 0:
    ts = breakouts.index[0]
    idx = df.index.get_loc(ts)
    
    print(f"\n📍 SHORT breakout at {ts.strftime('%H:%M')}")
    
    stops = calculate_stops_with_validity(df, idx, is_long=False)
    rr_results = calculate_best_rr_with_validity(stops, is_long=False, rr_desired=1.2)  # EXTREME vol
    
    print(f"\n--- Stop Validity ---")
    for stop_type in ['atr', 'swing', 'vwap']:
        valid = stops[f'{stop_type}_valid']
        rr = rr_results[stop_type]['rr']
        print(f"{stop_type.upper()}: valid={valid}, R:R={rr:.2f}")
    
    best_type, best_rr = select_best_valid_stop(rr_results)
    print(f"\nBest valid stop: {best_type.upper()} with R:R = {best_rr:.2f}")
    print(f"Min required: 1.5")
    print(f"\n{'✓ PASSES' if best_rr >= 1.5 else '✗ FAILS - SKIP'}")
    print(f"\nPine showed: SKIP 1.2:1 < 1.5 min")
else:
    print("No short breakout found - checking why...")
    potential = day_df[day_df['orb_complete'] & (day_df['close'] < day_df['orb_low'])]
    print(f"Bars closing below ORB: {len(potential)}")

=== R:R Gate Test: 2025-10-30 (SKIP trade) ===

📍 SHORT breakout at 09:35

--- Stop Validity ---
ATR: valid=True, R:R=1.20
SWING: valid=True, R:R=0.57
VWAP: valid=True, R:R=1.07

Best valid stop: ATR with R:R = 1.20
Min required: 1.5

✗ FAILS - SKIP

Pine showed: SKIP 1.2:1 < 1.5 min


In [ ]:
# Cell 9: Summary - Compare with TradingView

print("""
═══════════════════════════════════════════════════════════════
                    VALIDATION CHECKLIST v66
═══════════════════════════════════════════════════════════════

Fix #1: ORB State Machine
─────────────────────────
□ Breakout detected on breakout candle only (open in range, close out)
□ Entry bar still has pending=True (doesn't re-evaluate)
□ Phantom late entries blocked (don't open in range)
□ Re-breakouts work when price returns to range

Fix #2: Stop Validity
─────────────────────
□ VWAP wrong-side detected (VWAP above entry for LONG)
□ Invalid VWAP falls back to ATR
□ Min stop distance (15% ATR) enforced
□ No more 1-penny stops / 10k share positions

Test in TradingView:
────────────────────
1. Load v66 on NVDA
2. Check Oct 29 - LONG should execute (not vanish)
3. Check Oct 30 - SHORT should SKIP (R:R gate)
4. Find a bar where VWAP > price, verify no entry fires

═══════════════════════════════════════════════════════════════
""")

## Next Steps

Once Phase 1-2 re-validated:
- Phase 3 (Confluence) — should be mostly intact
- Phase 4 (Exits) — break-even, EMA exit, trailing
- Phase 5 (Adaptive) — vol regime adjustments